# 09. Kimi K3 — 93-layer text topology

93 layers에서 69 KDA layers와 24 Gated MLA layers를 사용하고, 첫 FFN은 dense, 이후 FFN은 896 experts / top-16 / 2 shared experts의 MoE를 사용한다. Attention Residual block size는 12다. Tensor width와 sequence/batch size는 작게 둔다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(11)
device = torch.device('cpu')
torch.set_num_threads(min(2, torch.get_num_threads()))
K3_FULL_ATTENTION_LAYERS = [4, 8, 12, 16, 20, 24, 28, 32, 36, 40, 44, 48, 52, 56, 60, 64, 68, 72, 76, 80, 84, 88, 92, 93]
K3_KDA_LAYERS = [layer_number for layer_number in range(1, 94) if layer_number not in K3_FULL_ATTENTION_LAYERS]
K3_DEPTH = 93
K3_HEADS = 96
K3_EXPERTS = 896
K3_TOP_K = 16
K3_SHARED_EXPERTS = 2
K3_ATTN_RES_BLOCK_SIZE = 12
assert len(K3_KDA_LAYERS) == 69
assert len(K3_FULL_ATTENTION_LAYERS) == 24


In [ ]:
class KimiRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        x_float = x.float()
        scale = torch.rsqrt(x_float.square().mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * scale).to(x.dtype) * self.weight

def kda_scan(q, k, v, g, beta, a_log, dt_bias, lower_bound=-5.0):
    batch, heads, length, head_dim = q.shape
    q = F.normalize(q, dim=-1)
    k = F.normalize(k, dim=-1)
    q = q * head_dim ** (-0.5)
    state = torch.zeros(batch, heads, head_dim, head_dim, dtype=q.dtype, device=q.device)
    outputs = []
    a = -torch.exp(a_log).view(1, heads, 1)
    dt_bias = dt_bias.view(1, heads, head_dim)
    for token_index in range(length):
        g_t = g[:, token_index]
        decay_log = a * F.softplus(g_t + dt_bias)
        if lower_bound is not None:
            decay_log = decay_log.clamp_min(lower_bound)
        alpha = decay_log.exp()
        beta_t = torch.sigmoid(beta[:, token_index])
        q_t = q[:, :, token_index]
        k_t = k[:, :, token_index]
        v_t = v[:, :, token_index]
        decayed = alpha[..., None] * state
        predicted = torch.einsum('bhkv,bhk->bhv', decayed, k_t)
        delta = v_t - predicted
        state = decayed + beta_t[..., None, None] * torch.einsum('bhk,bhv->bhkv', k_t, delta)
        outputs.append(torch.einsum('bhkv,bhk->bhv', state, q_t))
    return torch.stack(outputs, dim=2)

class KimiKDA(nn.Module):
    def __init__(self, model_dim=8, heads=96, head_dim=2, short_kernel=4):
        super().__init__()
        assert heads == 96
        assert short_kernel == 4
        self.heads = heads
        self.head_dim = head_dim
        self.short_kernel = short_kernel
        projection = heads * head_dim
        self.q_projection = nn.Linear(model_dim, projection, bias=False)
        self.k_projection = nn.Linear(model_dim, projection, bias=False)
        self.v_projection = nn.Linear(model_dim, projection, bias=False)
        self.q_conv = nn.Conv1d(projection, projection, short_kernel, groups=projection)
        self.k_conv = nn.Conv1d(projection, projection, short_kernel, groups=projection)
        self.v_conv = nn.Conv1d(projection, projection, short_kernel, groups=projection)
        self.f_a = nn.Linear(model_dim, head_dim, bias=False)
        self.f_b = nn.Linear(head_dim, projection, bias=False)
        self.a_log = nn.Parameter(torch.log(torch.empty(heads).uniform_(1.0, 16.0)))
        self.dt_bias = nn.Parameter(torch.zeros(projection))
        self.beta_projection = nn.Linear(model_dim, heads, bias=False)
        self.output_norm = KimiRMSNorm(head_dim)
        self.output_gate = nn.Linear(model_dim, projection, bias=False)
        self.output_projection = nn.Linear(projection, model_dim, bias=False)
    def _short_conv(self, x, convolution):
        x = x.transpose(1, 2)
        x = F.pad(x, (self.short_kernel - 1, 0))
        return F.silu(convolution(x)).transpose(1, 2)
    def _split_heads(self, x):
        batch, length, _ = x.shape
        return x.view(batch, length, self.heads, self.head_dim).transpose(1, 2)
    def forward(self, hidden):
        q = self._split_heads(self._short_conv(self.q_projection(hidden), self.q_conv))
        k = self._split_heads(self._short_conv(self.k_projection(hidden), self.k_conv))
        v = self._split_heads(self._short_conv(self.v_projection(hidden), self.v_conv))
        g = self.f_b(self.f_a(hidden)).view(hidden.size(0), hidden.size(1), self.heads, self.head_dim)
        beta = self.beta_projection(hidden)
        scanned = kda_scan(q, k, v, g, beta, self.a_log, self.dt_bias, lower_bound=-5.0)
        gate = torch.sigmoid(self.output_gate(hidden)).view(hidden.size(0), hidden.size(1), self.heads, self.head_dim)
        scanned = self.output_norm(scanned.transpose(1, 2)) * gate
        return self.output_projection(scanned.flatten(2))


## 2. Gated MLA with NoPE and q/k rotary-designated subspace


In [ ]:
class KimiGatedMLA(nn.Module):
    def __init__(self, model_dim=8, heads=96, q_rank=4, kv_rank=4, qk_nope_dim=2, qk_rope_dim=2, value_dim=2):
        super().__init__()
        assert heads == 96
        self.heads = heads
        self.qk_nope_dim = qk_nope_dim
        self.qk_rope_dim = qk_rope_dim
        self.value_dim = value_dim
        self.q_head_dim = qk_nope_dim + qk_rope_dim
        self.scaling = self.q_head_dim ** (-0.5)
        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_norm = KimiRMSNorm(q_rank)
        self.q_up = nn.Linear(q_rank, heads * self.q_head_dim, bias=False)
        self.kv_down_with_mqa = nn.Linear(model_dim, kv_rank + qk_rope_dim, bias=False)
        self.kv_norm = KimiRMSNorm(kv_rank)
        self.kv_up = nn.Linear(kv_rank, heads * (qk_nope_dim + value_dim), bias=False)
        self.output_gate = nn.Linear(model_dim, heads * value_dim, bias=False)
        self.output_projection = nn.Linear(heads * value_dim, model_dim, bias=False)
    def forward(self, hidden):
        batch, length, _ = hidden.shape
        q = self.q_up(self.q_norm(self.q_down(hidden))).view(batch, length, self.heads, self.q_head_dim).transpose(1, 2)
        q_pass, q_rot = q.split([self.qk_nope_dim, self.qk_rope_dim], dim=-1)
        compressed = self.kv_down_with_mqa(hidden)
        k_latent, k_rot = compressed.split([compressed.size(-1) - self.qk_rope_dim, self.qk_rope_dim], dim=-1)
        kv = self.kv_up(self.kv_norm(k_latent)).view(batch, length, self.heads, self.qk_nope_dim + self.value_dim).transpose(1, 2)
        k_pass, value = kv.split([self.qk_nope_dim, self.value_dim], dim=-1)
        k_rot = k_rot[:, None].expand(-1, self.heads, -1, -1)
        query = torch.cat([q_pass, q_rot], dim=-1)
        key = torch.cat([k_pass, k_rot], dim=-1)
        score = torch.matmul(query.float(), key.float().transpose(-2, -1)) * self.scaling
        causal = torch.ones(length, length, dtype=torch.bool, device=hidden.device).tril()
        score = score.masked_fill(~causal[None, None], float('-inf'))
        weight = score.softmax(dim=-1)
        attended = torch.matmul(weight, value.float()).to(hidden.dtype).transpose(1, 2).contiguous()
        gate = torch.sigmoid(self.output_gate(hidden)).view(batch, length, self.heads, self.value_dim)
        return self.output_projection((attended * gate).flatten(2))

mla_check = KimiGatedMLA()
assert mla_check.q_head_dim == mla_check.qk_nope_dim + mla_check.qk_rope_dim
assert mla_check.kv_down_with_mqa.out_features == 4 + mla_check.qk_rope_dim


## 3. SiTU-GLU, Stable LatentMoE, and Attention Residual


In [ ]:
def situ_gate(x, beta=4.0):
    return beta * torch.tanh(x / beta) * torch.sigmoid(x)

def situ_value(x, beta=25.0):
    return beta * torch.tanh(x / beta)

class DenseSiTUFFN(nn.Module):
    def __init__(self, model_dim=8, intermediate_dim=16):
        super().__init__()
        self.gate = nn.Linear(model_dim, intermediate_dim, bias=False)
        self.value = nn.Linear(model_dim, intermediate_dim, bias=False)
        self.output = nn.Linear(intermediate_dim, model_dim, bias=False)
    def forward(self, hidden):
        return self.output(situ_gate(self.gate(hidden)) * situ_value(self.value(hidden)))

class BatchedSiTUExperts(nn.Module):
    def __init__(self, experts, input_dim, intermediate_dim, output_dim):
        super().__init__()
        scale = 0.02
        self.gate_weight = nn.Parameter(scale * torch.randn(experts, input_dim, intermediate_dim))
        self.value_weight = nn.Parameter(scale * torch.randn(experts, input_dim, intermediate_dim))
        self.output_weight = nn.Parameter(scale * torch.randn(experts, intermediate_dim, output_dim))
    def selected_forward(self, hidden, expert_ids):
        gate_weight = self.gate_weight[expert_ids]
        value_weight = self.value_weight[expert_ids]
        output_weight = self.output_weight[expert_ids]
        gate = torch.einsum('bti,btkif->btkf', hidden, gate_weight)
        value = torch.einsum('bti,btkif->btkf', hidden, value_weight)
        intermediate = situ_gate(gate) * situ_value(value)
        return torch.einsum('btkf,btkfo->btko', intermediate, output_weight)

class KimiStableLatentMoE(nn.Module):
    def __init__(self, model_dim=8, latent_dim=4, routed_intermediate=4, shared_intermediate=8):
        super().__init__()
        self.experts = 896
        self.top_k = 16
        self.shared_expert_count = 2
        self.router = nn.Linear(model_dim, self.experts, bias=False)
        self.routing_bias = nn.Parameter(torch.zeros(self.experts), requires_grad=False)
        self.down = nn.Linear(model_dim, latent_dim, bias=False)
        self.routed = BatchedSiTUExperts(self.experts, latent_dim, routed_intermediate, latent_dim)
        self.latent_norm = KimiRMSNorm(latent_dim)
        self.up = nn.Linear(latent_dim, model_dim, bias=False)
        self.shared = BatchedSiTUExperts(self.shared_expert_count, model_dim, shared_intermediate, model_dim)
    def forward(self, hidden):
        raw_score = torch.sigmoid(self.router(hidden))
        expert_ids = (raw_score + self.routing_bias).topk(self.top_k, dim=-1).indices
        selected = raw_score.gather(-1, expert_ids)
        weight = selected / selected.sum(dim=-1, keepdim=True).clamp_min(1e-8)
        latent = self.down(hidden)
        routed = self.routed.selected_forward(latent, expert_ids)
        routed = (routed * weight[..., None]).sum(dim=2)
        routed = self.up(self.latent_norm(routed))
        shared_ids = torch.arange(self.shared_expert_count, device=hidden.device).view(1, 1, -1).expand(hidden.size(0), hidden.size(1), -1)
        shared = self.shared.selected_forward(hidden, shared_ids).sum(dim=2)
        return routed + shared

def attention_residual_mix(prefix_sum, bank, projection, norm):
    if bank.size(1) == 0:
        return prefix_sum
    values = torch.cat([bank, prefix_sum.unsqueeze(1)], dim=1)
    normalized = norm(values)
    score = projection(normalized).squeeze(-1)
    weight = score.softmax(dim=1)
    return torch.einsum('bk,bkd->bd', weight, values)


In [ ]:
class KimiK3Layer(nn.Module):
    def __init__(self, layer_index, model_dim=8):
        super().__init__()
        self.layer_index = layer_index
        layer_number = layer_index + 1
        self.attn_res_norm = KimiRMSNorm(model_dim)
        self.ffn_res_norm = KimiRMSNorm(model_dim)
        self.attn_res_proj = nn.Linear(model_dim, 1, bias=False)
        self.ffn_res_proj = nn.Linear(model_dim, 1, bias=False)
        self.input_norm = KimiRMSNorm(model_dim)
        self.post_attention_norm = KimiRMSNorm(model_dim)
        if layer_number in K3_FULL_ATTENTION_LAYERS:
            self.attention_kind = 'mla'
            self.attention = KimiGatedMLA(model_dim=model_dim)
        else:
            self.attention_kind = 'kda'
            self.attention = KimiKDA(model_dim=model_dim)
        if layer_index == 0:
            self.ffn_kind = 'dense'
            self.ffn = DenseSiTUFFN(model_dim=model_dim)
        else:
            self.ffn_kind = 'moe'
            self.ffn = KimiStableLatentMoE(model_dim=model_dim)
    def forward(self, hidden, bank):
        batch, length, dim = hidden.shape
        prefix = hidden
        flat_prefix = prefix.reshape(-1, dim)
        mixed = attention_residual_mix(flat_prefix, bank, self.attn_res_proj, self.attn_res_norm).view(batch, length, dim)
        if self.layer_index % K3_ATTN_RES_BLOCK_SIZE == 0:
            bank = torch.cat([bank, flat_prefix.unsqueeze(1)], dim=1)
            prefix = None
        attention_output = self.attention(self.input_norm(mixed))
        prefix = attention_output if prefix is None else prefix + attention_output
        ffn_input = attention_residual_mix(prefix.reshape(-1, dim), bank, self.ffn_res_proj, self.ffn_res_norm).view(batch, length, dim)
        ffn_input = self.post_attention_norm(ffn_input)
        ffn_output = self.ffn(ffn_input)
        return prefix + ffn_output, bank

class SmallWidthKimiK3(nn.Module):
    def __init__(self, model_dim=8):
        super().__init__()
        self.layers = nn.ModuleList([KimiK3Layer(index, model_dim) for index in range(K3_DEPTH)])
        self.output_res_norm = KimiRMSNorm(model_dim)
        self.output_res_proj = nn.Linear(model_dim, 1, bias=False)
        self.final_norm = KimiRMSNorm(model_dim)
    def forward(self, hidden):
        batch, length, dim = hidden.shape
        bank = hidden.new_zeros(batch * length, 0, dim)
        for layer in self.layers:
            hidden, bank = layer(hidden, bank)
        hidden = attention_residual_mix(hidden.reshape(-1, dim), bank, self.output_res_proj, self.output_res_norm).view(batch, length, dim)
        return self.final_norm(hidden), bank

model = SmallWidthKimiK3().to(device)
assert len(model.layers) == 93
assert sum(layer.attention_kind == 'kda' for layer in model.layers) == 69
assert sum(layer.attention_kind == 'mla' for layer in model.layers) == 24
assert model.layers[0].ffn_kind == 'dense'
assert all(layer.ffn_kind == 'moe' for layer in model.layers[1:])
assert all(layer.attention.heads == 96 for layer in model.layers)
assert all(layer.ffn.experts == 896 and layer.ffn.top_k == 16 and layer.ffn.shared_expert_count == 2 for layer in model.layers[1:])
hidden = torch.randn(1, 1, 8, device=device)
output, bank = model(hidden)
output.square().mean().backward()
assert bank.size(1) == 8
assert output.shape == hidden.shape
